# 07 · `create_agent` y el sistema de middleware

**Módulo 2 · Agentes** — *tiempo estimado: 1 h 45 min*

En el notebook anterior escribimos el bucle del agente a mano y vimos qué le faltaba: tope
de turnos, resumen del historial, aprobación humana, reintentos, métricas. Cada una de esas
cosas es un nodo o un envoltorio más, y añadirlas a mano convierte tu grafo en un plato de
espaguetis en el que la lógica del agente se pierde entre la fontanería.

**El middleware es la respuesta de LangChain 1.x a ese problema**, y es la diferencia
principal entre `create_agent` y el obsoleto `create_react_agent`. Es, con diferencia, la
parte más importante de este módulo.

Al terminar sabrás:

1. Los seis puntos de enganche del ciclo de un agente y qué se hace en cada uno.
2. Usar el middleware de fábrica, que cubre la mayoría de necesidades reales.
3. Escribir middleware propio con decoradores y con clases.
4. El orden de composición, que es donde se equivoca todo el mundo la primera vez.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                            if (p / "utils" / "curso.py").exists())))
from utils.curso import init, llm, mostrar_grafo, mostrar_mensajes, separador

init(proyecto="curso-langgraph-m2")

## 1. `create_agent` en dos minutos

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool

from utils.datos import tickets

df = tickets()


@tool(parse_docstring=True)
def contar_tickets(categoria: str = "todas", prioridad: str = "todas") -> str:
    """Cuenta tickets de soporte con los filtros dados.

    Args:
        categoria: facturacion, acceso_cuenta, bug_producto, integraciones, rendimiento,
            solicitud_funcionalidad, datos_privacidad, otros, o 'todas'.
        prioridad: baja, media, alta, critica, o 'todas'.
    """
    sel = df
    if categoria != "todas":
        sel = sel[sel.categoria == categoria]
    if prioridad != "todas":
        sel = sel[sel.prioridad == prioridad]
    return f"{len(sel)} tickets (categoría={categoria}, prioridad={prioridad})."


@tool(parse_docstring=True)
def resumen_por(dimension: str) -> str:
    """Resume el reparto de tickets por una dimensión.

    Args:
        dimension: categoria, prioridad, plan_cliente, canal o sentimiento.
    """
    if dimension not in {"categoria", "prioridad", "plan_cliente", "canal", "sentimiento"}:
        return "Error: dimensión no válida. Usa categoria, prioridad, plan_cliente, canal o sentimiento."
    conteo = df[dimension].value_counts()
    return f"Reparto por {dimension}:\n" + "\n".join(f"  {k}: {v}" for k, v in conteo.items())


HERRAMIENTAS = [contar_tickets, resumen_por]

agente = create_agent(
    model=llm(),
    tools=HERRAMIENTAS,
    system_prompt="Eres un analista de soporte. Usa las herramientas para responder con cifras exactas. "
                  "Responde en español y en 3 frases como máximo.",
)

salida = agente.invoke({"messages": [HumanMessage("¿Cómo se reparten los tickets por prioridad?")]},
                       {"recursion_limit": 20})
print(salida["messages"][-1].text)

Los parámetros que vas a usar de verdad:

| Parámetro | Para qué |
|---|---|
| `model` | Un modelo o `"openai:gpt-4o-mini"` |
| `tools` | Las herramientas |
| `system_prompt` | Instrucciones fijas |
| **`middleware`** | **La lista de comportamientos. El tema del notebook** |
| `response_format` | Esquema Pydantic para la salida final estructurada |
| `checkpointer` | Persistencia (módulo 3) |
| `store` | Memoria de largo plazo (módulo 3) |
| `context_schema` | Contexto de solo lectura de la petición |

## 2. Los seis puntos de enganche

El ciclo de un agente tiene seis costuras donde puedes meterte. Conocerlas es saber dónde va
cada cosa.

```
   entrada
      |
 [before_agent]        ......... una vez por ejecución, antes de todo
      |
      v
 +--> [before_model]   ......... antes de CADA llamada al modelo
 |       |
 |    ( wrap_model_call )  ..... ENVUELVE la llamada: puedes reintentar,
 |       |                       cambiar de modelo o cortocircuitar
 |       v
 |    [after_model]    ......... después de CADA respuesta del modelo
 |       |
 |       +-- ¿hay tool_calls? --> ( wrap_tool_call ) ---+
 |       |         no                  ENVUELVE cada       |
 |       v                             ejecución de        |
 |  [after_agent]      ..........      herramienta         |
 |       |             una vez, al final                   |
 |     salida                                              |
 +----------------------------------------------------------+
```

| Enganche | Cuándo | Casos típicos |
|---|---|---|
| `before_agent` | una vez, al empezar | Cargar memoria del usuario, validar la entrada |
| `before_model` | antes de cada llamada | Resumir el historial, inyectar contexto fresco |
| `wrap_model_call` | rodea la llamada | Reintentos, modelo alternativo, elegir modelo, medir |
| `after_model` | tras cada respuesta | Guardarraíles de salida, detección de PII, contadores |
| `wrap_tool_call` | rodea cada herramienta | Aprobación, reintentos, cachés, auditoría |
| `after_agent` | una vez, al terminar | Guardar memoria, emitir métricas finales |

La distinción clave: **`before_*` / `after_*` observan y modifican el estado**;
**`wrap_*` controla la ejecución** —puede llamar al *handler* varias veces, o ninguna.

## 3. Middleware de fábrica

Antes de escribir el tuyo, mira lo que ya existe. Cubre la mayor parte de lo que necesitarás.

| Middleware | Qué hace |
|---|---|
| `SummarizationMiddleware` | Resume el historial al pasar un umbral |
| `ModelCallLimitMiddleware` | Tope de llamadas al modelo por ejecución o por hilo |
| `ToolCallLimitMiddleware` | Tope de llamadas a herramientas, global o por herramienta |
| `ModelRetryMiddleware` | Reintenta la llamada al modelo con retroceso exponencial |
| `ToolRetryMiddleware` | Reintenta herramientas que fallan |
| `ModelFallbackMiddleware` | Si un modelo falla, prueba el siguiente |
| `ToolErrorMiddleware` | Convierte excepciones de herramienta en mensajes para el modelo |
| `PIIMiddleware` | Detecta y redacta datos personales en entrada, salida o resultados |
| `HumanInTheLoopMiddleware` | Pausa y pide aprobación antes de ciertas herramientas (módulo 3) |
| `LLMToolSelectorMiddleware` | Con muchas herramientas, preselecciona las relevantes |
| `ContextEditingMiddleware` | Poda resultados antiguos de herramientas del contexto |
| `TodoListMiddleware` | Da al agente una lista de tareas para planificar trabajos largos |

In [ ]:
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,
    PIIMiddleware,
    SummarizationMiddleware,
    ToolCallLimitMiddleware,
    ToolRetryMiddleware,
)

agente_robusto = create_agent(
    model=llm(),
    tools=HERRAMIENTAS,
    system_prompt="Eres un analista de soporte. Responde con cifras exactas, en español.",
    middleware=[
        # 1. Tope duro de llamadas al modelo. 'end' termina con lo que haya en vez de reventar.
        ModelCallLimitMiddleware(run_limit=6, exit_behavior="end"),
        # 2. Tope de llamadas a herramientas por ejecución.
        ToolCallLimitMiddleware(run_limit=8, exit_behavior="continue"),
        # 3. Reintento con retroceso ante fallos transitorios de herramienta.
        ToolRetryMiddleware(max_retries=2, backoff_factor=2.0, initial_delay=0.5),
        # 4. Si la conversación pasa de 3.000 tokens, se resume conservando los 10 últimos mensajes.
        SummarizationMiddleware(model=llm(), trigger=("tokens", 3000), keep=("messages", 10)),
        # 5. Los correos que aparezcan en la entrada se redactan antes de llegar al modelo.
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
    ],
)

salida = agente_robusto.invoke(
    {"messages": [HumanMessage(
        "Soy ana.lopez@acme.example. ¿Cuántos tickets críticos hay y cómo se reparten por canal?"
    )]},
    {"recursion_limit": 25},
)
mostrar_mensajes(salida, maximo=6)

Fíjate en el primer mensaje: el correo aparece redactado. `PIIMiddleware` lo interceptó
**antes** de que llegara al modelo, así que el dato personal nunca salió a la API del
proveedor. Eso es una medida de cumplimiento normativo implementada en una línea, y es un
argumento suficiente para usar middleware aunque no escribas ninguno propio.

In [ ]:
mostrar_grafo(agente_robusto)

En el diagrama ves los nodos que ha añadido cada middleware. Esa es la clave del sistema:
**el middleware no es magia, son nodos y envoltorios en el mismo grafo de LangGraph que ya
sabes leer.** Todo lo que aprendiste en el módulo 1 sigue aplicando.

## 4. Middleware propio con decoradores

Para casos de un solo enganche, los decoradores son la forma más corta.

### 4.1 `@before_model` — inyectar contexto fresco

In [ ]:
from datetime import datetime

from langchain.agents.middleware import after_model, before_model, dynamic_prompt, wrap_model_call, wrap_tool_call
from langchain.messages import AIMessage, SystemMessage


@before_model
def inyectar_fecha(state, runtime):
    """Añade la fecha actual antes de cada llamada.

    Un LLM no sabe qué día es. Si tu agente maneja plazos, SLA o "esta semana",
    esto elimina toda una familia de errores silenciosos.
    """
    ahora = datetime.now().strftime("%Y-%m-%d %H:%M")
    return {"messages": [SystemMessage(f"[contexto] Fecha y hora actuales: {ahora}.")]}


agente_fechado = create_agent(model=llm(), tools=HERRAMIENTAS, middleware=[inyectar_fecha])
salida = agente_fechado.invoke({"messages": [HumanMessage("¿Qué día es hoy?")]}, {"recursion_limit": 10})
print(salida["messages"][-1].text)

### 4.2 `@dynamic_prompt` — instrucciones que dependen del estado

In [ ]:
from dataclasses import dataclass


@dataclass
class ContextoUsuario:
    nombre: str
    nivel: str = "principiante"   # principiante | experto


@dynamic_prompt
def prompt_segun_nivel(request) -> str:
    """El prompt del sistema se calcula en cada llamada a partir del contexto y del estado."""
    ctx = request.runtime.context
    base = f"Eres un analista de soporte hablando con {ctx.nombre}. Responde en español."
    if ctx.nivel == "principiante":
        return base + " Explica cada cifra con una frase de contexto y evita la jerga."
    return base + " Sé telegráfico: solo cifras y nombres de campo, sin explicaciones."


agente_adaptativo = create_agent(
    model=llm(), tools=HERRAMIENTAS, context_schema=ContextoUsuario, middleware=[prompt_segun_nivel]
)

for nivel in ("principiante", "experto"):
    salida = agente_adaptativo.invoke(
        {"messages": [HumanMessage("¿Cuántos tickets críticos hay?")]},
        context=ContextoUsuario(nombre="Ana", nivel=nivel),
        config={"recursion_limit": 15},
    )
    print(f"[{nivel}] {salida['messages'][-1].text}\n")

### 4.3 `@wrap_model_call` — elegir el modelo según la dificultad

Este es el enganche más potente, porque **envuelve** la llamada: puedes ejecutarla varias
veces, cambiar la petición, o no ejecutarla en absoluto.

Un uso que se amortiza solo: mandar las preguntas fáciles a un modelo barato y reservar el
caro para las difíciles.

In [ ]:
import dataclasses

MODELO_BARATO = llm("gpt-4o-mini")
MODELO_CAPAZ = llm("gpt-4o")

SENALES_DIFICIL = ("compara", "analiza", "por qué", "tendencia", "recomienda", "evalúa", "correlación")


@wrap_model_call
def enrutar_modelo(request, handler):
    """Elige el modelo según la complejidad aparente de la conversación."""
    texto = " ".join(m.text for m in request.messages if m.type == "human").lower()
    dificil = len(texto) > 220 or any(s in texto for s in SENALES_DIFICIL)

    elegido = MODELO_CAPAZ if dificil else MODELO_BARATO
    print(f"    [enrutador] {'capaz' if dificil else 'barato'} ({len(texto)} caracteres)")

    # ModelRequest es un dataclass: se modifica con dataclasses.replace, sin mutarlo.
    return handler(dataclasses.replace(request, model=elegido))


agente_economico = create_agent(model=MODELO_BARATO, tools=HERRAMIENTAS, middleware=[enrutar_modelo])

for pregunta in ["¿Cuántos tickets hay de facturación?",
                 "Compara la distribución de prioridades entre planes y dime qué patrón ves."]:
    print(f"P: {pregunta}")
    salida = agente_economico.invoke({"messages": [HumanMessage(pregunta)]}, {"recursion_limit": 20})
    print(f"R: {salida['messages'][-1].text[:200]}\n")

### 4.4 `@wrap_tool_call` — auditar y proteger cada herramienta

In [ ]:
import time

AUDITORIA: list[dict] = []
HERRAMIENTAS_SENSIBLES = {"borrar_ticket", "reembolsar", "enviar_correo"}


@wrap_tool_call
def auditar_herramientas(request, handler):
    """Registra cada llamada y bloquea las sensibles sin autorización explícita."""
    nombre = request.tool_call["name"]

    if nombre in HERRAMIENTAS_SENSIBLES and not request.runtime.context.puede_escribir:
        from langchain.messages import ToolMessage
        return ToolMessage(
            f"Bloqueado: '{nombre}' requiere permisos de escritura y esta sesión es de solo lectura. "
            "Explícaselo al usuario y NO reintentes.",
            tool_call_id=request.tool_call["id"], status="error",
        )

    t0 = time.perf_counter()
    resultado = handler(request)
    AUDITORIA.append({"herramienta": nombre, "args": request.tool_call["args"],
                      "ms": round((time.perf_counter() - t0) * 1000, 1)})
    return resultado


@dataclass
class ContextoPermisos:
    puede_escribir: bool = False


agente_auditado = create_agent(
    model=llm(), tools=HERRAMIENTAS, context_schema=ContextoPermisos, middleware=[auditar_herramientas]
)

AUDITORIA.clear()
agente_auditado.invoke(
    {"messages": [HumanMessage("¿Cuántos tickets críticos hay y cómo se reparten por canal?")]},
    context=ContextoPermisos(puede_escribir=False),
    config={"recursion_limit": 20},
)

print("auditoría de esta ejecución:")
for r in AUDITORIA:
    print(f"  {r['herramienta']:<18} {r['args']}  ({r['ms']} ms)")

## 5. Middleware con clase: cuando hay estado propio

Los decoradores cubren un enganche. Cuando necesites **varios enganches coordinados** o
**claves nuevas en el estado**, hereda de `AgentMiddleware`.

El atributo `state_schema` es la parte interesante: el middleware **añade sus propias claves
al estado del agente**, sin que el agente que lo usa tenga que saberlo.

In [ ]:
import operator
from typing import Annotated

from langchain.agents.middleware import AgentMiddleware, AgentState


class EstadoConCoste(AgentState):
    """Claves que este middleware añade al estado del agente."""
    tokens_entrada: Annotated[int, operator.add]
    tokens_salida: Annotated[int, operator.add]
    llamadas_modelo: Annotated[int, operator.add]


PRECIO_POR_MILLON = {"gpt-4o-mini": (0.15, 0.60), "gpt-4o": (2.50, 10.00)}


class ContabilidadMiddleware(AgentMiddleware):
    """Acumula tokens y coste estimado de cada llamada al modelo.

    Es el ejemplo canónico de middleware con estado: mide algo transversal que ningún nodo
    del agente debería tener que conocer.
    """

    state_schema = EstadoConCoste

    def after_model(self, state, runtime) -> dict | None:
        ultimo = state["messages"][-1]
        uso = getattr(ultimo, "usage_metadata", None)
        if not uso:
            return None
        return {
            "tokens_entrada": uso.get("input_tokens", 0),
            "tokens_salida": uso.get("output_tokens", 0),
            "llamadas_modelo": 1,
        }

    @staticmethod
    def coste_euros(state, modelo: str = "gpt-4o-mini") -> float:
        entrada, salida = PRECIO_POR_MILLON[modelo]
        return (state["tokens_entrada"] * entrada + state["tokens_salida"] * salida) / 1_000_000


agente_contable = create_agent(
    model=llm(),
    tools=HERRAMIENTAS,
    system_prompt="Eres un analista de soporte. Responde en español con cifras exactas.",
    middleware=[ContabilidadMiddleware()],
)

salida = agente_contable.invoke(
    {"messages": [HumanMessage("Dame el reparto por categoría y por prioridad, y compáralos.")]},
    {"recursion_limit": 25},
)

print(salida["messages"][-1].text)
print(f"\nllamadas al modelo : {salida['llamadas_modelo']}")
print(f"tokens entrada     : {salida['tokens_entrada']:,}")
print(f"tokens salida      : {salida['tokens_salida']:,}")
print(f"coste estimado     : {ContabilidadMiddleware.coste_euros(salida):.6f} €")

Ese `coste estimado` es la métrica que nadie mide hasta que llega la factura. Con un
middleware de 15 líneas la tienes en **cada** ejecución, en el estado, lista para registrar,
para poner un tope o para facturar a quien corresponda.

## 6. El orden importa: primero de la lista = más externo

La composición del middleware es como la de los decoradores de Python: **el primero de la
lista envuelve a todos los demás**. Verlo una vez ahorra mucha confusión.

In [ ]:
def trazador(etiqueta: str):
    """Fabrica un wrap_model_call que anuncia cuándo entra y cuándo sale."""
    @wrap_model_call(name=f"trazador_{etiqueta}")
    def envoltorio(request, handler):
        print(f"    -> entra {etiqueta}")
        r = handler(request)
        print(f"    <- sale  {etiqueta}")
        return r
    return envoltorio


agente_trazado = create_agent(
    model=llm(), tools=[],
    middleware=[trazador("A"), trazador("B"), trazador("C")],
)

print("orden de la lista: [A, B, C]\n")
agente_trazado.invoke({"messages": [HumanMessage("Di 'hola' y nada más.")]}, {"recursion_limit": 5})

Anidamiento `A( B( C( modelo ) ) )`. De ahí salen las reglas prácticas:

- El middleware que debe verlo **todo** —auditoría, métricas, trazas— va **el primero**.
- El que debe estar **pegado al modelo** —elegir modelo, ajustar la petición— va **el último**.
- Un reintento en `A` reintenta también `B` y `C`. Si solo quieres reintentar la llamada
  cruda, el reintento va al final.
- `before_model` se ejecuta en el orden de la lista; `after_model`, en orden **inverso**,
  igual que al desapilar.

## 7. Salida estructurada con `response_format`

En el notebook 06 añadimos un nodo extra para formalizar la respuesta. `create_agent` lo hace
con un parámetro, y el resultado aparece en la clave `structured_response`.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class InformeSoporte(BaseModel):
    """Informe estructurado del analista de soporte."""

    titular: str = Field(description="La conclusión principal, en una frase")
    cifras: list[str] = Field(description="Cada cifra citada con su etiqueta, p. ej. '48 tickets críticos'")
    recomendacion: str = Field(description="Una acción concreta y accionable para el equipo")
    urgencia: Literal["baja", "media", "alta"] = Field(description="Urgencia de esa recomendación")


agente_informe = create_agent(
    model=llm(),
    tools=HERRAMIENTAS,
    system_prompt="Eres un analista de soporte. Usa las herramientas y basa todo en cifras reales.",
    response_format=InformeSoporte,
    middleware=[ModelCallLimitMiddleware(run_limit=6, exit_behavior="end")],
)

salida = agente_informe.invoke(
    {"messages": [HumanMessage("Analiza la cola de tickets y dime en qué debería centrarse el equipo.")]},
    {"recursion_limit": 25},
)

# `structured_response` puede FALTAR: si un tope de middleware corta la ejecución antes de
# que el agente llegue a la respuesta final, la clave nunca se escribe. Acceder con [] lanza
# un KeyError justo en el caso en que menos te apetece. En producción, siempre con .get().
informe = salida.get("structured_response")
if informe is None:
    print("sin respuesta estructurada: algún tope cortó la ejecución antes de terminar")
    print("texto libre:", salida["messages"][-1].text[:200])
else:
    print(f"titular      : {informe.titular}")
    print(f"urgencia     : {informe.urgencia}")
    print(f"recomendación: {informe.recomendacion}")
    print("cifras:")
    for c in informe.cifras:
        print("  -", c)

## 8. Ejercicios

> **EJERCICIO 7.1 — Un guardarraíl de salida**
>
> Escribe un middleware `@after_model` que compruebe la respuesta final del agente y, si
> contiene una promesa que el sistema no puede cumplir —"te devolveremos el dinero", "lo
> arreglamos hoy", "te llamamos en una hora"—, la sustituya por una versión neutra y deje
> constancia en el estado.
>
> Requisitos: solo debe actuar sobre respuestas finales (sin `tool_calls`), y la intervención
> tiene que quedar registrada para poder auditarla.

In [ ]:
# Tu solución aquí.

<details>
<summary><b>Ver solución 7.1</b></summary>

Dos detalles que hacen que este guardarraíl sea utilizable:

<ol>
<li><b>Solo actúa sobre respuestas finales.</b> Un <code>AIMessage</code> con
<code>tool_calls</code> es un paso intermedio; reescribirlo rompería el ciclo del agente y
dejaría llamadas sin respuesta.</li>
<li><b>Sustituye por <code>id</code>.</b> Devolver el mensaje corregido con el mismo
<code>id</code> hace que <code>add_messages</code> lo reemplace en su sitio. Si generases un
<code>id</code> nuevo, el historial acabaría con las dos versiones y el modelo vería la
prohibida.</li>
</ol>

Un guardarraíl léxico como este es tosco a propósito: es determinista, cuesta cero y no
falla. Para casos sutiles se usa un modelo clasificador como segunda capa, pero la primera
capa siempre debería ser barata.
</details>

In [ ]:
import re

# Los patrones van sobre la RAÍZ del verbo, no sobre el infinitivo: "solucionamos" no
# contiene "solucionar". Es el error más común al escribir guardarraíles léxicos en español,
# donde la conjugación cambia la terminación pero conserva la raíz.
PROMESAS_PROHIBIDAS = [
    (re.compile(r"\b(te|le|os)\s+\w*(devolv|devolver|reembols)\w*", re.I), "compromiso de reembolso"),
    (re.compile(r"\b(arregl|solucion|resolv|resuelv)\w*\s+(lo\s+)?\w{0,12}\s?"
                r"(hoy|mañana|en\s+\d+\s*(h|hora|minuto|día)\w*)", re.I), "compromiso de plazo"),
    (re.compile(r"\b(te|le|os)\s+\w*llamar\w*|\bte\s+llama\w*", re.I), "compromiso de llamada"),
    (re.compile(r"\bgarantiz\w*\b|\bte lo aseguro\b|\bsin ninguna duda\b", re.I), "garantía absoluta"),
]

# Comprobación rápida de que el guardarraíl distingue lo que debe distinguir.
_PRUEBAS = [
    ("Claro, te devolveremos el importe duplicado.", True),
    ("Lo solucionamos hoy mismo, no te preocupes.", True),
    ("Un compañero te llamará esta tarde.", True),
    ("Te garantizo que no volverá a pasar.", True),
    ("Nuestro horario de soporte es de 9 a 18.", False),
    ("He escalado tu caso al equipo correspondiente.", False),
]
for _texto, _esperado in _PRUEBAS:
    _detecta = any(p.search(_texto) for p, _ in PROMESAS_PROHIBIDAS)
    print(f"  {'ok ' if _detecta == _esperado else 'MAL'} {'bloquea' if _detecta else 'pasa   '} | {_texto}")

TEXTO_NEUTRO = (
    "He recogido tu caso y lo he pasado al equipo de soporte, que te responderá según el plazo "
    "de tu plan. No puedo comprometer plazos ni reembolsos concretos desde aquí."
)


class EstadoGuardarrail(AgentState):
    intervenciones: Annotated[list[str], operator.add]


@after_model(state_schema=EstadoGuardarrail)
def guardarrail_promesas(state, runtime):
    ultimo = state["messages"][-1]

    # Solo respuestas finales: un mensaje con tool_calls es un paso intermedio.
    if ultimo.type != "ai" or getattr(ultimo, "tool_calls", None):
        return None

    detectadas = [etiqueta for patron, etiqueta in PROMESAS_PROHIBIDAS if patron.search(ultimo.text)]
    if not detectadas:
        return None

    return {
        # Mismo id -> add_messages SUSTITUYE el mensaje, no añade otro.
        "messages": [AIMessage(TEXTO_NEUTRO, id=ultimo.id)],
        "intervenciones": [f"bloqueado: {', '.join(detectadas)} | original: {ultimo.text[:90]}"],
    }


agente_seguro = create_agent(
    model=llm(),
    tools=[],
    system_prompt="Eres un agente de atención al cliente muy servicial y resolutivo. "
                  "Responde en español, en 2 frases.",
    middleware=[guardarrail_promesas],
)

for mensaje in [
    "Me habéis cobrado dos veces, ¿me lo vais a devolver?",
    "¿Cuál es vuestro horario de soporte?",
]:
    salida = agente_seguro.invoke({"messages": [HumanMessage(mensaje)], "intervenciones": []},
                                  {"recursion_limit": 10})
    print(f"P: {mensaje}")
    print(f"R: {salida['messages'][-1].text}")
    print(f"   intervenciones: {salida['intervenciones'] or 'ninguna'}\n")

> **EJERCICIO 7.2 — Caché de herramientas**
>
> Escribe un `@wrap_tool_call` que cachee los resultados por `(nombre, argumentos)` durante
> la ejecución. Si el agente vuelve a pedir exactamente lo mismo, devuelve el resultado
> guardado sin ejecutar nada, y anótalo.
>
> Mide cuántas llamadas ahorras en una pregunta que provoque repeticiones.

In [ ]:
# Tu solución aquí.

<details>
<summary><b>Ver solución 7.2</b></summary>

La caché resuelve el <b>síntoma</b> del agente que da vueltas (el coste), no la
<b>causa</b> (que no sabe que ya lo preguntó). Sigue viendo el mismo resultado y sigue sin
avanzar. Por eso una caché es un buen complemento de un tope de llamadas, nunca un sustituto.

Y ojo con el alcance: esta caché vive en un diccionario del proceso, así que es <b>global</b>
y no se limpia. En producción quieres una caché por ejecución (en el estado) o con TTL
(<code>CachePolicy</code>, módulo 6), o acabarás sirviendo datos rancios entre usuarios.
</details>

In [ ]:
import json

CACHE: dict[str, str] = {}
ESTADISTICAS = {"aciertos": 0, "fallos": 0}


def clave_cache(llamada: dict) -> str:
    return f"{llamada['name']}::{json.dumps(llamada['args'], sort_keys=True, ensure_ascii=False)}"


@wrap_tool_call
def cachear(request, handler):
    from langchain.messages import ToolMessage

    clave = clave_cache(request.tool_call)
    if clave in CACHE:
        ESTADISTICAS["aciertos"] += 1
        return ToolMessage(f"{CACHE[clave]}\n[servido de caché]",
                           tool_call_id=request.tool_call["id"], name=request.tool_call["name"])

    ESTADISTICAS["fallos"] += 1
    resultado = handler(request)
    # Solo cacheamos ToolMessage: un Command trae actualizaciones de estado que no se pueden repetir.
    if isinstance(resultado, ToolMessage) and resultado.status != "error":
        CACHE[clave] = str(resultado.content)
    return resultado


agente_cacheado = create_agent(
    model=llm(), tools=HERRAMIENTAS,
    system_prompt="Eres un analista. Verifica cada cifra con una herramienta antes de usarla, "
                  "aunque creas recordarla.",
    middleware=[cachear],
)

CACHE.clear()
ESTADISTICAS.update(aciertos=0, fallos=0)

salida = agente_cacheado.invoke(
    {"messages": [HumanMessage(
        "¿Cuántos tickets críticos hay? Verifícalo, luego dime cuántos de facturación, "
        "y por último confírmame otra vez el número de críticos."
    )]},
    {"recursion_limit": 25},
)

print(salida["messages"][-1].text)
print(f"\ncaché: {ESTADISTICAS['aciertos']} aciertos, {ESTADISTICAS['fallos']} fallos "
      f"({len(CACHE)} entradas distintas)")

## 9. Resumen

- `create_agent` es el bucle del notebook 06 con las esquinas resueltas; lo que lo hace
  distinto es el **middleware**.
- Seis enganches: `before_agent`, `before_model`, `wrap_model_call`, `after_model`,
  `wrap_tool_call`, `after_agent`. Los `before/after` **observan y modifican estado**; los
  `wrap` **controlan la ejecución**.
- El middleware de fábrica cubre casi todo: topes, reintentos, resumen, PII, aprobación
  humana, selección de herramientas.
- Decoradores para un enganche; clase con `state_schema` cuando necesites claves propias o
  varios enganches coordinados.
- **Primero de la lista = más externo.** Auditoría al principio, selección de modelo al final.
- `ModelRequest` es un dataclass: modifícalo con `dataclasses.replace`, nunca mutándolo.
- `response_format` te da la salida estructurada en `structured_response`.

**Siguiente:** [`P2_proyecto_agente_analista.ipynb`](P2_proyecto_agente_analista.ipynb) — un
agente analista de datos completo, con middleware de producción y evaluado sobre preguntas
con respuesta conocida.